In [ ]:
# data github link: https://github.com/ContextScout/ned-graphs/blob/master/dataset/wikidata-disambig-test.json

In [1]:
# reading the json file


import json, pandas as pd
with open('wikidata-disambig-test.json') as f:
    data = json.load(f)


df = pd.DataFrame(data)
print('shape:', df.shape)
print()
print(df.head().to_string(max_colwidth=60))
print()
print('nulls:\n', df.isna().sum().to_string())
print('duplicate rows:', df.duplicated().sum())
print('unique mention strings:', df['string'].nunique())



shape: (10000, 4)

  correct_id       string                                                         text   wrong_id
0      Q3918   university  The University of Nanking (金陵大学) was a private universit...   Q7894492
1     Q16666      nanjing   private university in Nanjing, China which was founded ...    Q413681
2      Q1860      english   first school officially named University (in English) i...  Q15758838
3     Q81982  harpsichord   to the German-speaking nations. Silbermann, who worked ...  Q29385649
4   Q3758482  gary cooper   Bart van Oort and Gary Cooper. Opinions. Opinions about...   Q5524890

nulls:
 correct_id    0
string        0
text          0
wrong_id      0
duplicate rows: 2
unique mention strings: 5601


In [3]:
import json
import re
import time
from pathlib import Path

import requests

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
CACHE_FILE = Path("qid_status_cache.json")

session = requests.Session()
session.headers["User-Agent"] = "qid-validator/0.1 (educational; itisha)"


def _fetch_batch(batch: list[str], max_retries: int = 5) -> dict:
    """One wbgetentities call with retry/backoff on 429."""
    for attempt in range(max_retries):
        resp = session.get(WIKIDATA_API, params={
            "action": "wbgetentities",
            "ids": "|".join(batch),
            "props": "labels|descriptions",
            "languages": "en",
            "languagefallback": 1,  # items whose label lives under "mul" still resolve
            "format": "json",
        }, timeout=30)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 2 ** attempt * 5))
            print(f"rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()["entities"]
    raise RuntimeError(f"still rate-limited after {max_retries} retries")


def check_qids(qids: list[str]) -> dict[str, dict]:
    # load cache: QIDs checked in a previous run are never re-queried
    cache = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}

    unique = list(dict.fromkeys(str(q) for q in qids))
    # sentinels like <NIL>, <NO_MAPPING>, OTHER are not Wikidata items
    queryable = [q for q in unique if re.fullmatch(r"Q\d+", q)]
    todo = [q for q in queryable if q not in cache]
    print(f"{len(unique)} unique ids, {len(queryable)} queryable, {len(todo)} not yet cached")

    for i in range(0, len(todo), 50):
        batch = todo[i : i + 50]
        entities = _fetch_batch(batch)

        # a redirected qid comes back keyed by its target id, with a redirects.from field
        redirect_from = {e.get("redirects", {}).get("from"): k
                         for k, e in entities.items() if e.get("redirects")}
        for qid in batch:
            final = redirect_from.get(qid, qid)
            ent = entities.get(qid) or entities.get(final)
            if not ent or "missing" in ent:
                cache[qid] = {"valid": False, "label": None, "description": None,
                              "redirects_to": None, "url": None}
            else:
                cache[qid] = {
                    "valid": True,
                    "label": ent.get("labels", {}).get("en", {}).get("value"),
                    "description": ent.get("descriptions", {}).get("en", {}).get("value"),
                    "redirects_to": final if final != qid else None,
                    "url": f"https://www.wikidata.org/wiki/{ent.get('id', qid)}",
                }

        # save progress after every batch, so an interruption loses nothing
        CACHE_FILE.write_text(json.dumps(cache))
        time.sleep(1.5)  # stay under the rate limit

    return {q: cache[q] for q in queryable}



# validate the gold entity QIDs
status = check_qids(df["correct_id"].tolist())

def _field(col: str):
    return df["correct_id"].map(lambda q: status.get(str(q), {}).get(col))

df["qid_valid"] = _field("valid").fillna(False).astype(bool)
df["wd_label"]       = _field("label")
df["wd_description"] = _field("description")
df["wd_url"]         = _field("url")


df

5727 unique ids, 5727 queryable, 5727 not yet cached
rate limited, waiting 35s...
rate limited, waiting 38s...
rate limited, waiting 37s...
rate limited, waiting 25s...
rate limited, waiting 39s...
rate limited, waiting 38s...
rate limited, waiting 38s...
rate limited, waiting 39s...
rate limited, waiting 38s...
rate limited, waiting 38s...
rate limited, waiting 39s...


,correct_id,string,text,wrong_id,qid_valid,wd_label,wd_description,wd_url
0,Q3918,university,The University of Nanking (金陵大学) was a private...,Q7894492,True,university,an institution of tertiary education and resea...,https://www.wikidata.org/wiki/Q3918
1,Q16666,nanjing,"private university in Nanjing, China which wa...",Q413681,True,Nanjing,"capital city of Jiangsu Province, China, forme...",https://www.wikidata.org/wiki/Q16666
2,Q1860,english,first school officially named University (in ...,Q15758838,True,English,West Germanic language,https://www.wikidata.org/wiki/Q1860
3,Q81982,harpsichord,"to the German-speaking nations. Silbermann, w...",Q29385649,True,harpsichord,plucked-string keyboard instrument,https://www.wikidata.org/wiki/Q81982
4,Q3758482,gary cooper,Bart van Oort and Gary Cooper. Opinions. Opin...,Q5524890,True,Gary Cooper,British musician,https://www.wikidata.org/wiki/Q3758482
...,...,...,...,...,...,...,...,...
9995,Q81292,acre,= 100 sq ft). ≈0.00002295684 acres. 92 903.04...,Q126084,True,acre,unit of area,https://www.wikidata.org/wiki/Q81292
9996,Q190,god,that all beings are expressions of and part o...,Q5575809,True,God,principal object of faith in monotheistic reli...,https://www.wikidata.org/wiki/Q190
9997,Q193291,spirit,idealistic and panentheistic philosophies. RS...,Q12002312,True,spirit,immaterial being,https://www.wikidata.org/wiki/Q193291
9998,Q1845,bible,as Religious Science or Science of Mind. Upon...,Q2900994,True,Bible,collection of sacred books in Judaism and Chri...,https://www.wikidata.org/wiki/Q1845


In [4]:
# Stratification: ambiguous vs unambiguous mentions

amb = (df.groupby("string")
       .agg(n_mentions=("correct_id", "size"),
            n_qids=("correct_id", "nunique"),
            qids=("correct_id", lambda x: sorted(set(x))))
       .query("n_qids > 1")
       .sort_values("n_qids", ascending=False))
amb


amb

,n_mentions,n_qids,qids
string,,,
newport,4,3,"[Q1012011, Q101254, Q54264]"
liberal party,3,3,"[Q23060, Q500190, Q622441]"
dallas,4,3,"[Q1003667, Q16557, Q168708]"
capital,6,3,"[Q193893, Q5119, Q8137]"
melbourne,23,3,"[Q3141, Q3304955, Q79459]"
...,...,...,...
erie,2,2,"[Q234061, Q3056522]"
engineer,2,2,"[Q128124, Q81096]"
emperor,3,2,"[Q264029, Q39018]"


In [5]:
# Balanced eval set: equal numbers of ambiguous and non-ambiguous rows
df["ambiguous"] = df.groupby("string")["correct_id"].transform("nunique") > 1

amb_rows    = df[df["ambiguous"]]
nonamb_rows = df[~df["ambiguous"]]
n = min(len(amb_rows), len(nonamb_rows))   # the ambiguous side is the scarce one

balanced_df = (pd.concat([
        amb_rows,                                      # all 475 ambiguous rows
        nonamb_rows.sample(n=n, random_state=42),      # 475 random non-ambiguous
    ])
    .sample(frac=1, random_state=42)                   # shuffle
    .reset_index(drop=True))

print(balanced_df["ambiguous"].value_counts())
balanced_df.head()

ambiguous
True     475
False    475
Name: count, dtype: int64


,correct_id,string,text,wrong_id,qid_valid,wd_label,wd_description,wd_url,ambiguous
0,Q2966,heidelberg,became a professor of philosophy at Heidelber...,Q1594002,True,Heidelberg,"large city in Baden-Württemberg, Germany",https://www.wikidata.org/wiki/Q2966,True
1,Q6607,guitar,"7"" EP Dope, Guns 'n' Fucking in the Streets V...",Q17214008,True,guitar,fretted string instrument,https://www.wikidata.org/wiki/Q6607,False
2,Q35666,glacier,"glacier, which was much larger during the last...",Q5566122,True,glacier,large persistent body of ice,https://www.wikidata.org/wiki/Q35666,False
3,Q47528,radar,The AI.24 (Foxhunter) airborne radar is carried,Q37483717,True,radar,object detection system based on radio waves,https://www.wikidata.org/wiki/Q47528,False
4,Q2320,february 6,"David Hart (born February 6, 1954) in Marianna...",Q5439852,True,February 6,day of the year,https://www.wikidata.org/wiki/Q2320,False


In [6]:
balanced_df.shape

(950, 9)

In [16]:
MODELS = ['en_core_web_sm','en_core_web_md','en_core_web_lg']

def covered(m, terms):
    return m in terms or any(m in t for t in terms)

def extraction_flags(data, model_name):
    nlp = spacy.load(model_name)
    mentions = data['string'].str.lower().str.strip()
    rows = []
    for m, doc in zip(mentions, nlp.pipe(data['text'], batch_size=128)):
        ents   = {e.text.lower().strip() for e in doc.ents}
        chunks = {c.text.lower().strip() for c in doc.noun_chunks} | {c.root.text.lower().strip() for c in doc.noun_chunks}
        nouns  = {t.text.lower() for t in doc if t.pos_ in ('NOUN','PROPN')}
        rows.append({'ner': covered(m, ents), 'noun_chunks': covered(m, chunks), 'pos_nouns': covered(m, nouns)})
    rep = pd.DataFrame(rows, index=data.index)
    rep['any_route'] = rep.any(axis=1)
    return rep

def extraction_report(data, models=MODELS):
    results = {}
    for name in models:
        t0 = time.perf_counter()
        rep = extraction_flags(data, name)
        row = rep.mean()
        row['amb']     = rep.loc[data['ambiguous'].values, 'any_route'].mean()
        row['non-amb'] = rep.loc[~data['ambiguous'].values, 'any_route'].mean()
        row['seconds'] = time.perf_counter() - t0
        results[name] = row
    report = pd.DataFrame(results).T.round(3)
    report['seconds'] = report['seconds'].round(1)
    report = report.rename(columns={'any_route': 'any route'})
    report.index.name = 'model'
    return report

print(extraction_report(balanced_df).to_string())

                  ner  noun_chunks  pos_nouns  any route    amb  non-amb  seconds
model                                                                            
en_core_web_sm  0.669        0.934      0.718      0.982  0.987    0.977      5.5
en_core_web_md  0.657        0.946      0.727      0.984  0.994    0.975      6.6
en_core_web_lg  0.674        0.936      0.722      0.985  0.987    0.983      7.7


In [18]:
# en_core_web_sm pre-processing per sentence: NER entities, POS tags, noun
# phrases, lemmas — then filter to rows where the gold mention is recovered by NER
nlp_sm = spacy.load("en_core_web_sm")

mentions = balanced_df["string"].str.lower().str.strip()
records = []
for m, doc in zip(mentions, nlp_sm.pipe(balanced_df["text"], batch_size=128)):
    ents = [(e.text, e.label_) for e in doc.ents]
    ent_texts = {e.text.lower().strip() for e in doc.ents}
    # the entity span that covers the gold mention, and its NER type
    ner_mention = next((t for t, _ in ents
                        if m == t.lower().strip() or m in t.lower()), None)
    ner_label = next((l for t, l in ents if t == ner_mention), None)
    records.append({
        "sm_ner_extracted": covered(m, ent_texts),
        "ner_mention":      ner_mention,
        "ner_label":        ner_label,                          # PERSON, GPE, ORG, ...
        "ner_entities":     ents,                               # all entities in the sentence
        "noun_phrases":     [c.text for c in doc.noun_chunks],
        "pos_tags":         [(t.text, t.pos_) for t in doc if not t.is_space],
        "context_lemmas":   [t.lemma_.lower() for t in doc
                             if t.is_alpha and not t.is_stop],  # for later embedding/overlap scoring
    })

prep = pd.DataFrame(records, index=balanced_df.index)
balanced_df[prep.columns] = prep    # idempotent: safe to re-run the cell

ner_df = balanced_df[balanced_df["sm_ner_extracted"]].reset_index(drop=True)

print(f"kept {len(ner_df)} / {len(balanced_df)} rows "
      f"({len(ner_df)/len(balanced_df):.1%})")
print(ner_df["ambiguous"].value_counts().rename("rows").to_string())
ner_df.head()

kept 636 / 950 rows (66.9%)
ambiguous
True     336
False    300


,correct_id,string,text,wrong_id,qid_valid,wd_label,wd_description,wd_url,ambiguous,sm_ner_extracted,ner_mention,ner_label,ner_entities,noun_phrases,pos_tags,context_lemmas
0,Q2966,heidelberg,became a professor of philosophy at Heidelber...,Q1594002,True,Heidelberg,"large city in Baden-Württemberg, Germany",https://www.wikidata.org/wiki/Q2966,True,True,Heidelberg,ORG,"[(Heidelberg, ORG), (The Science of Logic, WOR...","[a professor, philosophy, Heidelberg, The Scie...","[(became, VERB), (a, DET), (professor, NOUN), ...","[professor, philosophy, heidelberg, science, l..."
1,Q2320,february 6,"David Hart (born February 6, 1954) in Marianna...",Q5439852,True,February 6,day of the year,https://www.wikidata.org/wiki/Q2320,False,True,"February 6, 1954",DATE,"[(David Hart, PERSON), (February 6, 1954, DATE...","[David Hart, Marianna, Florida, an actor, his ...","[(David, PROPN), (Hart, PROPN), ((, PUNCT), (b...","[david, hart, bear, february, marianna, florid..."
2,Q458,european union,"is not compulsory. In addition, while UK, Iri...",Q319328,True,European Union,political and economic union of 27 European st...,https://www.wikidata.org/wiki/Q458,False,True,European Union,ORG,"[(UK, GPE), (Irish, NORP), (Commonwealth, ORG)...","[addition, UK, Irish and Commonwealth citizens...","[(is, AUX), (not, PART), (compulsory, ADJ), (....","[compulsory, addition, uk, irish, commonwealth..."
3,Q1860,english,Anniversary edition. Stanford: Stanford UP. I...,Q5378143,True,English,West Germanic language,https://www.wikidata.org/wiki/Q1860,False,True,English,LANGUAGE,"[(Anniversary, PERSON), (Stanford, ORG), (Stan...","[ Anniversary edition, Stanford, Stanford UP, ...","[(Anniversary, PROPN), (edition, NOUN), (., PU...","[anniversary, edition, stanford, stanford, isb..."
4,Q1196400,mega man x,and Tengu Man. Unlike previous Mega Man games...,Q14632819,True,Mega Man X,1993 video game,https://www.wikidata.org/wiki/Q1196400,False,True,Mega Man X,PERSON,"[(Tengu Man, PERSON), (Mega, PERSON), (first, ...","[ and Tengu Man, previous Mega Man games, the ...","[(and, CCONJ), (Tengu, PROPN), (Man, PROPN), (...","[tengu, man, unlike, previous, mega, man, game..."


In [19]:
ner_df.to_csv("dataset_3_sen_level.csv", index=False)